In [1]:
!pip install -q gradio numpy pandas matplotlib

import gradio as gr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def atm_density_kg_m3(alt_km, solar_activity="Moderate"):
  # Simplified exponential scale-height layers in LEO (approx. 180-900 km)
  h = np.asarray(alt_km, dtype=float)
  scale_mult = {
      "Low Solar Activity": 0.70,
      "Moderate": 1.00,
      "High Solar Activity (Stormy)": 1.65,
  }.get(solar_activity, 1.00)

  # Base reference density & scale height near 400-550 km
  rho_400 = 3.5e-12 * scale_mult
  H_km = 58.0 if solar_activity != "High Solar Activity (Stormy)" else 66.0
  rho = rho_400 * np.exp(-(h - 400.0) / H_km)
  # Smooth floor for very high LEO
  return np.maximum(rho, 1e-16)


def simulate_decay_curve(start_alt_km, mass_kg, area_m2, cd, solar_activity, max_years=10.0):
  mu = 398600.4418e9  # m^3 / s^2
  re_m = 6378137.0
  bc = (cd * area_m2) / max(mass_kg, 0.1)  # Ballistic coefficient (m^2/kg)

  dt_days = 5.0
  dt_sec = dt_days * 86400.0
  n_steps = int((max_years * 365.25) / dt_days) + 1

  days = np.arange(n_steps) * dt_days
  alts_km = np.full(n_steps, np.nan)
  h_m = start_alt_km * 1000.0
  alts_km[0] = start_alt_km

  reentry_day = None
  for k in range(1, n_steps):
    cur_alt_km = h_m / 1000.0
    if cur_alt_km <= 120.0:
      reentry_day = days[k - 1]
      alts_km[k - 1] = 120.0
      break
    r_m = re_m + h_m
    v_mps = np.sqrt(mu / r_m)
    rho = atm_density_kg_m3(cur_alt_km, solar_activity)
    # Orbital energy decay: da/dt ≈ -2 * a * (drag_acc / v), drag_acc = 0.5 * rho * v^2 * bc
    dh_dt_mps = -rho * v_mps * bc * r_m
    h_m = max(120000.0, h_m + dh_dt_mps * dt_sec)
    alts_km[k] = h_m / 1000.0

  return days / 365.25, alts_km, reentry_day, bc


def run_decay_simulation(alt_a, alt_b, alt_c, alt_d, mass_kg, area_m2, cd, solar_activity, horizon_years):
  alts_list = sorted([float(alt_a), float(alt_b), float(alt_c), float(alt_d)])
  bc = (float(cd) * float(area_m2)) / max(float(mass_kg), 0.1)

  plt.close("all")
  plt.style.use("dark_background")
  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 4.8), dpi=110)
  fig.patch.set_facecolor("#0b0f17")

  colors = ["#f87171", "#fbbf24", "#38bdf8", "#34d399"]
  table_rows = []
  lifetimes_yr = []

  for idx, h0 in enumerate(alts_list):
    yrs, h_curve, re_day, _ = simulate_decay_curve(
        h0, mass_kg, area_m2, cd, solar_activity, max_years=float(horizon_years)
    )
    ax1.plot(
        yrs,
        h_curve,
        color=colors[idx % len(colors)],
        linewidth=2.2,
        label=f"{h0:.0f} km start",
    )

    if re_day is not None:
      life_yr = re_day / 365.25
      status_str = f"Re-enters in {life_yr:.2f} yr ({re_day:.0f} d)"
      end_alt_str = "120.0 km (re-entry)"
    else:
      # Extrapolate rough remaining lifetime if still above 120 km at horizon
      valid_mask = ~np.isnan(h_curve)
      last_h = float(h_curve[valid_mask][-1])
      drop_total = max(h0 - last_h, 0.05)
      rate_km_per_yr = drop_total / float(horizon_years)
      est_yr = (h0 - 120.0) / max(rate_km_per_yr, 1e-4)
      life_yr = min(est_yr, 99.9)
      status_str = f">{horizon_years:.0f} yr (est. ~{est_yr:.1f} yr)"
      end_alt_str = f"{last_h:.1f} km"

    lifetimes_yr.append(min(life_yr, float(horizon_years) * 1.1))
    table_rows.append({
        "Initial Altitude": f"{h0:.0f} km",
        "Altitude at Horizon": end_alt_str,
        "Lifetime Assessment": status_str,
        "Design Implication": (
            "Short lifetime — frequent reboost or de-orbit margin"
            if h0 < 420
            else (
                "Balanced LEO operational window; monitor drag"
                if h0 <= 550
                else "Longer dwell; consider post-mission disposal / debris rules"
            )
        ),
    })

  ax1.axhline(120.0, color="#94a3b8", linestyle=":", linewidth=1.2, label="Re-entry (~120 km)")
  ax1.set_title(
      f"Orbital Decay Across Altitudes ({solar_activity})",
      color="#f8fafc",
      fontsize=11,
      fontweight="bold",
  )
  ax1.set_xlabel("Elapsed Time (Years)", color="#94a3b8", fontsize=9)
  ax1.set_ylabel("Altitude (km)", color="#94a3b8", fontsize=9)
  ax1.tick_params(colors="#94a3b8", labelsize=8)
  ax1.grid(True, alpha=0.18, color="#64748b")
  ax1.legend(loc="lower left", fontsize=8, facecolor="#121826", edgecolor="#263248")
  ax1.set_facecolor("#111724")

  bar_labels = [f"{h:.0f} km" for h in alts_list]
  bars = ax2.bar(bar_labels, lifetimes_yr, color=colors[: len(alts_list)], width=0.5)
  ax2.set_title(
      f"Lifetime Comparison (Capped at {horizon_years:.0f} yr Plot)",
      color="#f8fafc",
      fontsize=11,
      fontweight="bold",
  )
  ax2.set_ylabel("Years (Plot Horizon)", color="#94a3b8", fontsize=9)
  ax2.tick_params(colors="#94a3b8", labelsize=8)
  ax2.grid(True, axis="y", alpha=0.18, color="#64748b")
  ax2.set_facecolor("#111724")
  for b, val in zip(bars, lifetimes_yr):
    ax2.text(
        b.get_x() + b.get_width() / 2.0,
        val + max(lifetimes_yr) * 0.02,
        f"{val:.1f}y",
        ha="center",
        va="bottom",
        color="#f8fafc",
        fontsize=8.5,
        fontweight="bold",
    )

  plt.tight_layout(pad=1.5)

  cards_html = f"""
    <div style="width:100%;box-sizing:border-box;font-family:Inter,system-ui,sans-serif;color:#e2e8f0;">
      <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:10px;background:#121826;border:1px solid #263248;border-radius:12px;padding:14px 18px;margin-bottom:12px;">
        <div>
          <div style="font-size:11px;color:#38bdf8;font-weight:700;letter-spacing:0.5px;">LEO ATMOSPHERIC DRAG &amp; ORBITAL DECAY SIMULATION</div>
          <div style="font-size:20px;font-weight:700;color:#f8fafc;margin-top:2px;">{alts_list[0]:.0f} / {alts_list[1]:.0f} / {alts_list[2]:.0f} / {alts_list[3]:.0f} km <span style="font-size:13px;color:#94a3b8;">({solar_activity})</span></div>
        </div>
        <span style="background:#0f293a;color:#38bdf8;border:1px solid #0284c7;padding:6px 12px;border-radius:999px;font-size:12px;font-weight:700;">BC: {bc:.4f} m²/kg</span>
      </div>

      <div style="display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:10px;">
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">SPACECRAFT MASS</div>
          <div style="font-size:19px;font-weight:700;color:#38bdf8;margin-top:4px;">{float(mass_kg):.1f} kg</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">DRAG AREA (A)</div>
          <div style="font-size:19px;font-weight:700;color:#a78bfa;margin-top:4px;">{float(area_m2):.3f} m²</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">DRAG COEFF (Cd)</div>
          <div style="font-size:19px;font-weight:700;color:#34d399;margin-top:4px;">{float(cd):.2f}</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">SOLAR REGIME</div>
          <div style="font-size:16px;font-weight:700;color:#fbbf24;margin-top:4px;">{solar_activity.split(' ')[0]}</div>
        </div>
      </div>
    </div>
    """

  constraints_html = f"""
    <div style="width:100%;box-sizing:border-box;font-family:Inter,system-ui,sans-serif;background:#111724;border:1px solid #263248;border-radius:12px;padding:16px 20px;line-height:1.6;font-size:14px;color:#cbd5e1;">
      <div style="font-weight:700;font-size:15px;color:#f8fafc;margin-bottom:8px;">📐 Design Constraints Created by Orbital Decay</div>
      <ul style="margin:0;padding-left:20px;">
        <li><b>Mission Lifetime vs. Altitude Trade-Off:</b> Lower altitudes experience exponentially higher thermospheric density, causing faster decay and shortening mission life unless propellant/reboost is budgeted; higher altitudes extend life but increase radiation exposure and end-of-life disposal obligations.</li>
        <li><b>Solar Array Area vs. Drag Area Conflict:</b> Larger deployable solar panels increase power generation, but they also increase cross-sectional drag area (<b>A</b>) and ballistic coefficient (<b>C_d·A/m</b>), accelerating altitude loss—especially when panels are not edge-aligned during high-drag phases.</li>
        <li><b>Propulsion, Attitude &amp; Orbit-Maintenance Budget:</b> Maintaining a fixed imaging/revisit altitude or ground-track repeat cycle under <b>{solar_activity}</b> conditions requires delta-v propellant mass, ADCS pointing margins, and ground-track drift compensation.</li>
      </ul>
    </div>
    """

  return cards_html, fig, constraints_html, pd.DataFrame(table_rows)


full_width_css = """
html, body, .gradio-container, .gradio-container > .main, .gradio-container > .main > .wrap, .block, .form {
  background-color: #0b0f17 !important;
  color: #e2e8f0 !important;
  max-width: 100% !important;
  width: 100% !important;
  margin: 0 !important;
  box-sizing: border-box !important;
}
.gradio-container {
  padding: 10px 14px !important;
}
.table-wrap, .dataframe {
  width: 100% !important;
  max-width: 100% !important;
  overflow-x: visible !important;
}
"""

with gr.Blocks() as demo:
  gr.Markdown("## 📉 Orbital Decay Simulator & Satellite Design Constraints")

  with gr.Row(equal_height=False):
    with gr.Column(scale=4, min_width=320):
      alt_a = gr.Slider(250, 750, value=380, step=10, label="Altitude 1 (km)")
      alt_b = gr.Slider(250, 750, value=450, step=10, label="Altitude 2 (km)")
      alt_c = gr.Slider(250, 750, value=520, step=10, label="Altitude 3 (km)")
      alt_d = gr.Slider(250, 750, value=620, step=10, label="Altitude 4 (km)")
      mass_sl = gr.Slider(1.0, 50.0, value=4.0, step=0.5, label="Spacecraft Mass (kg)")
      area_sl = gr.Slider(0.01, 1.50, value=0.06, step=0.01, label="Cross-Sectional Area (m²)")
      cd_sl = gr.Slider(1.5, 3.5, value=2.2, step=0.1, label="Drag Coefficient (Cd)")
      solar_dd = gr.Dropdown(
          choices=["Low Solar Activity", "Moderate", "High Solar Activity (Stormy)"],
          value="Moderate",
          label="Solar Activity Level",
      )
      horizon_sl = gr.Slider(1, 25, value=10, step=1, label="Simulation Horizon (Years)")
      run_btn = gr.Button("⚡ Simulate Orbital Decay", variant="primary")

    with gr.Column(scale=8, min_width=460):
      out_cards = gr.HTML()
      out_plot = gr.Plot(label="Altitude Decay Curves & Lifetime Comparison")

  with gr.Row():
    with gr.Column(scale=12):
      out_constraints = gr.HTML()
      out_table = gr.Dataframe(
          label="Multi-Altitude Decay & Design Constraint Table", interactive=False
      )

  run_btn.click(
      fn=run_decay_simulation,
      inputs=[alt_a, alt_b, alt_c, alt_d, mass_sl, area_sl, cd_sl, solar_dd, horizon_sl],
      outputs=[out_cards, out_plot, out_constraints, out_table],
  )

demo.launch(share=False, theme=gr.themes.Soft(), css=full_width_css)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>